In [5]:
import numpy as np
import json
from openai import OpenAI
import pypdf
import os
import math
from dotenv import load_dotenv
from pathlib import Path

In [9]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print(api_key is not None)

model_name = "gpt-5-mini"
embedding_model = "text-embedding-3-small"
client = OpenAI(api_key=api_key)

True


In [7]:
saved_chunks = json.load(open("guidewire_chunks.json", "r"))

In [8]:
def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot_product / (norm_a * norm_b)

def search_chunks_by_embedding(chunks, query, top_k=5):
    query_response = client.embeddings.create(
        model=embedding_model,
        input=query
    )
    query_embedding = query_response.data[0].embedding

    results = []
    for chunk in chunks:
        score = cosine_similarity(query_embedding, chunk["embedding"])
        results.append({
            "file_name": chunk["file_name"],
            "page_number": chunk["page_number"],
            "text": chunk["text"],
            "score": score
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

In [13]:
company_name = "Guidewire"
workflow_name = "Risk Register"

WORKFLOW_QUERIES = {
    "Risk Register": [
        f"{company_name} risk factors competition customer concentration pricing pressure churn retention",
        f"{company_name} operational risk implementation cloud migration execution regulatory legal",
        f"{company_name} revenue margin growth slowdown contract renewal profitability demand uncertainty"
    ],
    
    "Investment Memo": [
        f"{company_name} company overview products customers revenue business model segments geography",
        f"{company_name} market growth drivers competition differentiation positioning",
        f"{company_name} risks opportunities strategy expansion profitability"
    ]
}

In [14]:
def gather_workflow_evidence(chunks, workflow_name, top_k_per_query=5, max_context_chunks=12):
    workflow_queries = WORKFLOW_QUERIES.get(workflow_name, [])
    all_results = []

    for query in workflow_queries:
        results = search_chunks_by_embedding(chunks, query, top_k=top_k_per_query)
        all_results.extend(results)

    deduped = {}

    for result in all_results:
        key = (result["file_name"], result["page_number"], result["text"])

        if key not in deduped or result["score"] > deduped[key]["score"]:
            deduped[key] = result

    final_results = list(deduped.values())
    final_results.sort(key=lambda x: x["score"], reverse=True)
    final_results = final_results[:max_context_chunks]

    return {
        "workflow": workflow_name,
        "queries": workflow_queries,
        "results": final_results
    }


In [ ]:
evidence = gather_workflow_evidence(saved_chunks, workflow_name)

In [ ]:
def build_context(results, max_chars=18000):
    context = ""
    for result in results:
        entry = f"File: {result['file_name']}, Page: {result['page_number']}\n{result['text']}\n\n"
        if len(context) + len(entry) > max_chars:
            break
        context += entry
    return context  